In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.ml.feature import Imputer
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [2]:
spark = SparkSession.builder.appName('OlistData').getOrCreate()

26/02/18 08:18:30 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
hdfs_path = '/data/olist/'

In [4]:
customers_df = spark.read.csv(hdfs_path + 'olist_customers_dataset.csv', header=True, inferSchema = True)
geolocation_df = spark.read.csv(hdfs_path + 'olist_geolocation_dataset.csv', header=True, inferSchema = True)
order_item_df = spark.read.csv(hdfs_path + 'olist_order_items_dataset.csv', header=True, inferSchema = True)
payments_df = spark.read.csv(hdfs_path + 'olist_order_payments_dataset.csv', header=True, inferSchema = True)
reviews_df = spark.read.csv(hdfs_path + 'olist_order_reviews_dataset.csv', header=True, inferSchema = True)
orders_df = spark.read.csv(hdfs_path + 'olist_orders_dataset.csv', header=True, inferSchema = True)
products_df = spark.read.csv(hdfs_path + 'olist_products_dataset.csv', header=True, inferSchema = True)
sellers_df = spark.read.csv(hdfs_path + 'olist_sellers_dataset.csv', header=True, inferSchema = True)
category_translation_df = spark.read.csv(hdfs_path + 'product_category_name_translation.csv', header=True, inferSchema = True)

In [5]:
## Cache frequently used data for Better Performance
customers_df.cache()
orders_df.cache()
order_item_df.cache()

DataFrame[order_id: string, order_item_id: int, product_id: string, seller_id: string, shipping_limit_date: timestamp, price: double, freight_value: double]

## Joins without optimization

In [6]:
## Joining datasets

orders_items_joined_df = orders_df.join(order_item_df, 'order_id', 'inner')
order_items_products_df = orders_items_joined_df.join(products_df, 'product_id', 'inner')
orders_items_products_sellers_df = order_items_products_df.join(sellers_df, 'seller_id', 'inner')
full_orders_df = orders_items_products_sellers_df.join(customers_df, 'customer_id', 'inner')

## Geolocation dataset
full_orders_df = full_orders_df.join(geolocation_df, full_orders_df.customer_zip_code_prefix == geolocation_df.geolocation_zip_code_prefix, 'left')

## Reviews dataset
full_orders_df = full_orders_df.join(reviews_df, 'order_id', 'left')

## Payments dataset
full_orders_df = full_orders_df.join(payments_df, 'order_id', 'left')

In [8]:
## Caching the joined dataset
full_orders_df.cache()

26/02/17 07:06:52 WARN CacheManager: Asked to cache already cached data.


DataFrame[order_id: string, customer_id: string, seller_id: string, product_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, order_item_id: int, shipping_limit_date: timestamp, price: double, freight_value: double, product_category_name: string, product_name_lenght: int, product_description_lenght: int, product_photos_qty: int, product_weight_g: int, product_length_cm: int, product_height_cm: int, product_width_cm: int, seller_zip_code_prefix: int, seller_city: string, seller_state: string, customer_unique_id: string, customer_zip_code_prefix: int, customer_city: string, customer_state: string, geolocation_zip_code_prefix: int, geolocation_lat: double, geolocation_lng: double, geolocation_city: string, geolocation_state: string, review_id: string, review_score: string, review_comment_title: string, review_commen

In [18]:
full_orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

## Optimized Joins for data integration

In [6]:
## Joining datasets

orders_items_joined_df = orders_df.join(order_item_df, 'order_id', 'inner')
order_items_products_df = orders_items_joined_df.join(products_df, 'product_id', 'inner')
orders_items_products_sellers_df = order_items_products_df.join(broadcast(sellers_df), 'seller_id', 'inner')
full_orders_df = orders_items_products_sellers_df.join(customers_df, 'customer_id', 'inner')

## Geolocation dataset
full_orders_df = full_orders_df.join(broadcast(geolocation_df), full_orders_df.customer_zip_code_prefix == geolocation_df.geolocation_zip_code_prefix, 'left')

## Reviews dataset
full_orders_df = full_orders_df.join(broadcast(reviews_df), 'order_id', 'left')

## Payments dataset
full_orders_df = full_orders_df.join(payments_df, 'order_id', 'left')

In [7]:
full_orders_df.cache()

26/02/18 08:20:10 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame[order_id: string, customer_id: string, seller_id: string, product_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, order_item_id: int, shipping_limit_date: timestamp, price: double, freight_value: double, product_category_name: string, product_name_lenght: int, product_description_lenght: int, product_photos_qty: int, product_weight_g: int, product_length_cm: int, product_height_cm: int, product_width_cm: int, seller_zip_code_prefix: int, seller_city: string, seller_state: string, customer_unique_id: string, customer_zip_code_prefix: int, customer_city: string, customer_state: string, geolocation_zip_code_prefix: int, geolocation_lat: double, geolocation_lng: double, geolocation_city: string, geolocation_state: string, review_id: string, review_score: string, review_comment_title: string, review_commen

In [8]:
## Total Orders per customer

customer_order_count_df = full_orders_df.groupBy('customer_id')\
.agg(count('order_id').alias('total_orders'))\
.orderBy('total_orders', ascending = False)

customer_order_count_df.show(5)

+--------------------+------------+
|         customer_id|total_orders|
+--------------------+------------+
|351e40989da90e704...|       11427|
|50920f8cd0681fd86...|       10752|
|9b43e2a62de9bab3a...|        8556|
|270c23a11d024a44c...|        8001|
|5c87184371002d49e...|        6876|
+--------------------+------------+
only showing top 5 rows



In [9]:
## Average review score per seller

average_review_per_customer = full_orders_df.groupBy('seller_id').agg(avg('review_score').alias('average_review_score'))\
.orderBy('average_review_score', ascending = False)
average_review_per_customer.show(5)

+--------------------+--------------------+
|           seller_id|average_review_score|
+--------------------+--------------------+
|a61cc04793308395a...|                 5.0|
|4aba6a02a788d3ec8...|                 5.0|
|05a48cc8859962767...|                 5.0|
|33ab10be054370c25...|                 5.0|
|c74f14c1e26cf1bd5...|                 5.0|
+--------------------+--------------------+
only showing top 5 rows



In [10]:
## Top 10 most sold products

top_products_df = full_orders_df.groupBy('product_id')\
.agg(count('order_id').alias('products_sold'))\
.orderBy('products_sold', ascending = False).limit(10)

top_products_df.show()

+--------------------+-------------+
|          product_id|products_sold|
+--------------------+-------------+
|aca2eb7d00ea1a7b8...|        86740|
|422879e10f4668299...|        81110|
|99a4788cb24856965...|        78775|
|389d119b48cf3043d...|        60248|
|d1c427060a0f73f6b...|        59274|
|368c6c730842d7801...|        58358|
|53759a2ecddad2bb8...|        52654|
|53b36df67ebb7c415...|        52105|
|154e7e31ebfa09220...|        42700|
|3dd2a17168ec895c7...|        40787|
+--------------------+-------------+



In [11]:
## Top 10 customer by Spending
from pyspark.sql.types import *

customers_spending = full_orders_df.groupBy('customer_id').agg(sum('payment_value').cast(DecimalType(15, 2))\
                                                               .alias('customers_spending'))\
                                                                .orderBy('customers_spending', ascending = False).limit(10)
customers_spending.show()

+--------------------+------------------+
|         customer_id|customers_spending|
+--------------------+------------------+
|1ff773612ab8934db...|       17568252.00|
|05455dfa7cd02f13d...|       13282083.36|
|ec5b2ba62e5743423...|       10388528.64|
|0c792d32a3251b4f6...|        8254681.60|
|78fc46047c4a639e8...|        7488520.00|
|1617b1357756262bf...|        7433259.52|
|1dbc055ccab23ed89...|        7216273.40|
|d5f2b3f597c7ccafb...|        6800018.12|
|dd3f1762eb601f41c...|        6746388.48|
|10de381f8a8d23fff...|        5184499.50|
+--------------------+------------------+



In [12]:
## Total Revenue Per Seller

seller_revenue_df = full_orders_df.groupBy('seller_id').agg(sum('price').cast(DecimalType(15, 2)).alias('total_seller_revenue'))
seller_revenue_df.show(5)

+--------------------+--------------------+
|           seller_id|total_seller_revenue|
+--------------------+--------------------+
|8e6cc767478edae94...|          1145757.40|
|4d600e08ecbe08258...|           436434.23|
|9b1050e85becf3ae9...|            19667.34|
|cb5ff1b9715e99589...|            13235.00|
|038b75b729c8a9a04...|            17979.50|
+--------------------+--------------------+
only showing top 5 rows



## Window function and Ranking

In [13]:
## Defining window specification
window_spec = Window.partitionBy('seller_id').orderBy(desc('price'))

In [14]:
## Rank Top selling products per seller

top_selling_products_df = full_orders_df.withColumn('rank', rank().over(window_spec)).filter(col('rank')<=5)

top_selling_products_df.select('seller_id', 'price', 'rank').show(5)

+--------------------+-----+----+
|           seller_id|price|rank|
+--------------------+-----+----+
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
+--------------------+-----+----+
only showing top 5 rows



In [15]:
## Dense Rank for Sellers Based on Revenue

sellers_revenue = full_orders_df.groupBy('seller_id').agg(sum('payment_value').cast(DecimalType(15, 2)).alias('total_revenue'))

window_spec = Window.orderBy(desc('total_revenue'))

ranked_sellers = sellers_revenue.withColumn('rank', dense_rank().over(window_spec))

top_5_sellers = ranked_sellers.filter(col('rank') <= 5)
top_5_sellers.show()

26/02/18 08:23:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/18 08:23:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/18 08:23:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/18 08:23:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/18 08:23:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/18 08:23:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/18 0

+--------------------+-------------+----+
|           seller_id|total_revenue|rank|
+--------------------+-------------+----+
|7c67e1448b00f6e96...|  85683597.56|   1|
|da8622b14eb17ae28...|  53811434.33|   2|
|1025f0e2d44d7041d...|  50434674.68|   3|
|4a3ca9315b744ce9f...|  48727295.52|   4|
|1f50f920176fa81da...|  44570278.14|   5|
+--------------------+-------------+----+



26/02/18 08:23:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/18 08:23:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


## Advance data Aggregation

In [16]:
## Total Revenue and Average Order Value (AOV) per customer

customer_spending_df = full_orders_df.groupBy('customer_id')\
.agg(
    count('order_id').alias('total_orders'),
    sum('price').alias('total_spent'),
    round(avg('price'),2).alias('AOV')
).orderBy(desc('total_spent'))

customer_spending_df.show(5)

+--------------------+------------+-----------+------+
|         customer_id|total_orders|total_spent|   AOV|
+--------------------+------------+-----------+------+
|d3e82ccec3cb5f956...|        6876|  6662844.0| 969.0|
|df55c14d1476a9a34...|         743|  3565657.0|4799.0|
|fe5113a38e3575c04...|        2292|  3293604.0|1437.0|
|ec5b2ba62e5743423...|        1428|  2556120.0|1790.0|
|63b964e79dee32a35...|        6072|  2501664.0| 412.0|
+--------------------+------------+-----------+------+
only showing top 5 rows



In [17]:
## Seller performance Metrics (Revenue, Average Review, Order Count)

full_orders_df.groupBy('seller_id')\
.agg(
    count('order_id').alias('order_count'),
    sum('price').cast(DecimalType(15, 2)).alias('total_revenue'),
    round(avg('review_score'), 2).alias('average_review_score'),
    round(stddev('price'), 2).alias('price_variability')
).orderBy(desc('total_revenue')).show(10)

+--------------------+-----------+-------------+--------------------+-----------------+
|           seller_id|order_count|total_revenue|average_review_score|price_variability|
+--------------------+-----------+-------------+--------------------+-----------------+
|4869f7a5dfa277a7d...|     184587|  36138717.32|                4.09|           111.65|
|53243585a1d6dc264...|      54514|  34291592.95|                4.12|           499.65|
|4a3ca9315b744ce9f...|     330661|  33759570.84|                3.77|            59.37|
|7c67e1448b00f6e96...|     233306|  32282321.79|                3.42|            50.39|
|fa1c13f2614d7b5c4...|      87686|  30139386.31|                4.38|            307.7|
|da8622b14eb17ae28...|     264433|  29857669.73|                3.98|            72.92|
|7e93a43ef30c4f03f...|      50226|  26315706.30|                4.15|           377.24|
|1025f0e2d44d7041d...|     229587|  22937518.52|                3.89|             84.3|
|46dc3b2cc0980fb8e...|      9042

In [18]:
## Product Polularity Metrics

product_metrics = full_orders_df.groupBy('product_id')\
.agg(
    count('order_id').alias('total_sales'),
    sum('price').alias('total_revenue'),
    round(avg('price'), 2).alias('avg_price'),
    round(stddev('price'),2).alias('price_volatility'),
    collect_set('seller_id').alias('unique_sellers')
).orderBy(desc('total_sales'))

product_metrics.show(10)

+--------------------+-----------+-----------------+---------+----------------+--------------------+
|          product_id|total_sales|    total_revenue|avg_price|price_volatility|      unique_sellers|
+--------------------+-----------+-----------------+---------+----------------+--------------------+
|aca2eb7d00ea1a7b8...|      86740|6164630.299996104|    71.07|            3.17|[955fee9216a65b61...|
|422879e10f4668299...|      81110|4442791.509997975|    54.77|            4.46|[1f50f920176fa81d...|
|99a4788cb24856965...|      78775|6921762.709996365|    87.87|            4.08|[4a3ca9315b744ce9...|
|389d119b48cf3043d...|      60248| 3280533.12999878|    54.45|            4.37|[1f50f920176fa81d...|
|d1c427060a0f73f6b...|      59274|8220103.330003085|   138.68|           16.58|[a1043bafd471dff5...|
|368c6c730842d7801...|      58358|3181698.899999021|    54.52|            4.59|[1f50f920176fa81d...|
|53759a2ecddad2bb8...|      52654|2893017.499999484|    54.94|            4.52|[1f50f920176

In [19]:
## Monthly Revenue and Order Count Trend

order_count_trend = full_orders_df.groupBy(year('order_purchase_timestamp').alias('year'), month('order_purchase_timestamp').alias('month'))\
.agg(
    countDistinct('order_id').alias('orders_count'),
    sum('price').cast(DecimalType(15, 2)).alias('total_revenue'),
    round(avg('price'), 2).alias('avg_order_value'),
    min('price').alias('min_order_value'),
    max('price').alias('max_order_value')
).orderBy(desc('total_revenue'))

order_count_trend.show()

+----+-----+------------+-------------+---------------+---------------+---------------+
|year|month|orders_count|total_revenue|avg_order_value|min_order_value|max_order_value|
+----+-----+------------+-------------+---------------+---------------+---------------+
|2017|   11|        7451| 166037217.54|         116.74|            3.9|         2990.0|
|2018|    4|        6934| 158851337.12|          125.5|           0.85|        3399.99|
|2018|    5|        6853| 156856056.59|          126.8|            3.9|         3930.0|
|2018|    3|        7188| 154576832.52|         117.44|           4.99|        4099.99|
|2018|    1|        7220| 154037882.12|         114.89|           4.78|         3690.0|
|2018|    2|        6694| 141066599.15|         114.11|           2.99|         3099.9|
|2018|    6|        6160| 140667502.12|         125.22|            3.5|         4590.0|
|2018|    7|        6273| 138919305.66|         127.82|            3.0|         6729.0|
|2018|    8|        6452| 135340

In [20]:
## Customer Retention Analysis (First & Last Order)

full_orders_df.groupBy('customer_unique_id')\
.agg(
    min('order_purchase_timestamp').alias('first_order_date'),
    max('order_purchase_timestamp').alias('last_order_date'),
    countDistinct('order_id').alias('total_orders'),
    round(avg('price'), 2).alias('AOV')
).orderBy(desc('total_orders')).show()

+--------------------+-------------------+-------------------+------------+------+
|  customer_unique_id|   first_order_date|    last_order_date|total_orders|   AOV|
+--------------------+-------------------+-------------------+------------+------+
|8d50f5eadf50201cc...|2017-05-15 23:30:03|2018-08-20 19:14:26|          16|  45.6|
|3e43e6105506432c9...|2017-09-18 18:53:15|2018-02-27 18:36:39|           9| 76.09|
|1b6c7548a2a1f9037...|2017-11-13 16:44:41|2018-02-14 13:22:12|           7| 85.52|
|ca77025e7201e3b30...|2017-10-09 12:34:39|2018-06-01 11:38:29|           7| 67.22|
|6469f99c1f9dfae77...|2017-09-19 01:02:44|2018-06-28 00:43:34|           7|  73.8|
|47c1a3033b8b77b3a...|2017-08-07 14:14:22|2018-01-24 15:15:26|           6|  88.7|
|f0e310a6839dce9de...|2017-05-20 08:53:30|2018-04-05 09:04:45|           6| 73.01|
|12f5d6e1cbf93dafd...|2017-01-05 14:18:03|2017-01-05 15:25:10|           6|  9.73|
|63cfc61cee11cbe30...|2017-05-11 14:39:53|2018-05-28 17:20:02|           6| 52.71|
|dc8

## Advance Enrichment

In [21]:
## Order status flags

full_orders_df = full_orders_df.withColumn('is_delivered', when(col('order_status') == 'delivered', lit(1)).otherwise(lit(0)))\
.withColumn('is_cancelled', when(col('order_status') == 'canceled', lit(1)).otherwise(lit(0)))

In [22]:
full_orders_df.where(full_orders_df['order_status'] == 'canceled').select('order_status', 'is_delivered', 'is_cancelled').show(10)

+------------+------------+------------+
|order_status|is_delivered|is_cancelled|
+------------+------------+------------+
|    canceled|           0|           1|
|    canceled|           0|           1|
|    canceled|           0|           1|
|    canceled|           0|           1|
|    canceled|           0|           1|
|    canceled|           0|           1|
|    canceled|           0|           1|
|    canceled|           0|           1|
|    canceled|           0|           1|
|    canceled|           0|           1|
+------------+------------+------------+
only showing top 10 rows



In [23]:
## Order revenue calculation

full_orders_df = full_orders_df.withColumn('order_revenue', col('price') + col('freight_value'))

In [24]:
full_orders_df.select('price', 'freight_value', 'order_revenue').show(5)

+-----+-------------+-------------+
|price|freight_value|order_revenue|
+-----+-------------+-------------+
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
+-----+-------------+-------------+
only showing top 5 rows



In [25]:
## Customer segmentation based on spending

customer_spending_df = customer_spending_df.withColumn('customer_segment', when(col('AOV') >= 1200, "High-Value")\
                                .when((col('AOV') < 1200) & (col('AOV') >= 700), "Medium-Value")\
                                .otherwise("Low-Value"))

customer_spending_df.show(10)

+--------------------+------------+------------------+-------+----------------+
|         customer_id|total_orders|       total_spent|    AOV|customer_segment|
+--------------------+------------+------------------+-------+----------------+
|d3e82ccec3cb5f956...|        6876|         6662844.0|  969.0|    Medium-Value|
|df55c14d1476a9a34...|         743|         3565657.0| 4799.0|      High-Value|
|fe5113a38e3575c04...|        2292|         3293604.0| 1437.0|      High-Value|
|ec5b2ba62e5743423...|        1428|         2556120.0| 1790.0|      High-Value|
|63b964e79dee32a35...|        6072|         2501664.0|  412.0|       Low-Value|
|46bb3c0b1a65c8399...|         748|         2336752.0| 3124.0|      High-Value|
|05455dfa7cd02f13d...|        2184| 2160194.400000087|  989.1|    Medium-Value|
|3690e975641f01bd0...|         802|         2124498.0| 2649.0|      High-Value|
|349509b216bd5ec11...|         743|         1923627.0| 2589.0|      High-Value|
|695476b5848d64ba0...|         687|18205

In [26]:
full_orders_df = full_orders_df.join(customer_spending_df.select('customer_id', 'customer_segment'), 'customer_id', how = 'left')

In [27]:
full_orders_df.select('customer_id', 'customer_segment').show(5)

+--------------------+----------------+
|         customer_id|customer_segment|
+--------------------+----------------+
|3ce436f183e68e078...|       Low-Value|
|3ce436f183e68e078...|       Low-Value|
|3ce436f183e68e078...|       Low-Value|
|3ce436f183e68e078...|       Low-Value|
|3ce436f183e68e078...|       Low-Value|
+--------------------+----------------+
only showing top 5 rows



In [28]:
## Hourly order distribution

full_orders_df = full_orders_df.withColumn('hour_of_day', expr('hour(order_purchase_timestamp)'))

full_orders_df.select('order_purchase_timestamp', 'hour_of_day').show(5)

+------------------------+-----------+
|order_purchase_timestamp|hour_of_day|
+------------------------+-----------+
|     2017-09-13 08:59:02|          8|
|     2017-09-13 08:59:02|          8|
|     2017-09-13 08:59:02|          8|
|     2017-09-13 08:59:02|          8|
|     2017-09-13 08:59:02|          8|
+------------------------+-----------+
only showing top 5 rows



In [29]:
## Weekday Vs Weekend orders

full_orders_df = full_orders_df.withColumn('order_day_type', when(dayofweek('order_purchase_timestamp')\
                                                 .isin(1, 7), lit('Weekend')).otherwise('Weekday'))

full_orders_df.select('order_purchase_timestamp', 'order_day_type').show(5)

+------------------------+--------------+
|order_purchase_timestamp|order_day_type|
+------------------------+--------------+
|     2017-09-13 08:59:02|       Weekday|
|     2017-09-13 08:59:02|       Weekday|
|     2017-09-13 08:59:02|       Weekday|
|     2017-09-13 08:59:02|       Weekday|
|     2017-09-13 08:59:02|       Weekday|
+------------------------+--------------+
only showing top 5 rows



In [30]:
## A new Column freight category --> frieght_value (low, medium, high)

full_orders_df = full_orders_df.withColumn('freight_category', when(col('freight_value') >= 300 , lit("high"))\
                          .when((col('freight_value') < 300) & (col('freight_value') >= 100) , lit("medium"))\
                         .otherwise("low"))

full_orders_df.select('freight_value', 'freight_category').show(5)

+-------------+----------------+
|freight_value|freight_category|
+-------------+----------------+
|        13.29|             low|
|        13.29|             low|
|        13.29|             low|
|        13.29|             low|
|        13.29|             low|
+-------------+----------------+
only showing top 5 rows



In [31]:
## Order Volume by Geolocation state

order_volume_state = full_orders_df.groupBy('geolocation_state')\
.agg(countDistinct('order_id').alias('order_volume')).orderBy(desc('order_volume'))

order_volume_state.show()

+-----------------+------------+
|geolocation_state|order_volume|
+-----------------+------------+
|               SP|       41360|
|               RJ|       12749|
|               MG|       11534|
|               RS|        5439|
|               PR|        4987|
|               SC|        3625|
|               BA|        3350|
|               ES|        2019|
|               GO|        1998|
|               DF|        1961|
|               PE|        1644|
|               CE|        1323|
|               PA|         967|
|               MT|         901|
|               MA|         736|
|               MS|         709|
|               PB|         530|
|               PI|         490|
|               RN|         480|
|               AL|         410|
+-----------------+------------+
only showing top 20 rows



In [32]:
!hdfs dfs -ls /data/olist/processed

In [33]:
## Saving the dataframe as a parquet

full_orders_df.write.mode('overwrite').parquet('/data/olist/processed')

In [34]:
!hdfs dfs -ls -h /data/olist/processed

Found 11 items
-rw-r--r--   2 root hadoop          0 2026-02-18 08:25 /data/olist/processed/_SUCCESS
-rw-r--r--   2 root hadoop     31.6 M 2026-02-18 08:25 /data/olist/processed/part-00000-16b1628c-87f6-4c10-bedd-95f804aadf47-c000.snappy.parquet
-rw-r--r--   2 root hadoop     31.2 M 2026-02-18 08:25 /data/olist/processed/part-00001-16b1628c-87f6-4c10-bedd-95f804aadf47-c000.snappy.parquet
-rw-r--r--   2 root hadoop     31.0 M 2026-02-18 08:25 /data/olist/processed/part-00002-16b1628c-87f6-4c10-bedd-95f804aadf47-c000.snappy.parquet
-rw-r--r--   2 root hadoop     30.9 M 2026-02-18 08:25 /data/olist/processed/part-00003-16b1628c-87f6-4c10-bedd-95f804aadf47-c000.snappy.parquet
-rw-r--r--   2 root hadoop     31.4 M 2026-02-18 08:25 /data/olist/processed/part-00004-16b1628c-87f6-4c10-bedd-95f804aadf47-c000.snappy.parquet
-rw-r--r--   2 root hadoop     18.2 M 2026-02-18 08:25 /data/olist/processed/part-00005-16b1628c-87f6-4c10-bedd-95f804aadf47-c000.snappy.parquet
-rw-r--r--   2 root hadoop   

In [35]:
spark.stop()